# Drawing line arrangements

Beautiful plots of (real) line arrangements in the projective chart $z=1$.

* Intersection points are computed **exactly** (sympy rationals / real quadratic fields), then embedded for drawing; multiple points ($m\ge 3$) are colored by multiplicity.
* By default the chart is **whitened**: an affine change of coordinates spreads the multiple-point cloud isotropically (projectively harmless, much prettier for slope-anisotropic coordinates). Pass `whiten=False` for raw coordinates.
* Lines at infinity ($a=b=0$) cannot appear in this chart and are counted in the title; complex-field arrangements (e.g. dual Hesse over $\mathbb{Q}(\sqrt{-3})$) have no real chart and raise.


In [ ]:
import glob, json
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from novelty import arrangement_from_record
from arrangement import LineArrangement, ProjectiveLine

MULT_COLORS = {3: "#4878a8", 4: "#2e8b57", 5: "#e08214", 6: "#c0392b",
               7: "#7b3294", 8: "#542788"}


def load_by_hash(prefix, roots=("results_from_HPC", "results_local",
                                "results_penalized_saito")):
    """Load a certified arrangement by lattice-hash prefix."""
    for root in roots:
        for p in glob.glob(f"{root}/**/certified.jsonl", recursive=True):
            for line in open(p):
                try:
                    r = json.loads(line)
                except ValueError:
                    continue
                if r.get("lattice_hash", "").startswith(prefix):
                    return arrangement_from_record(r), r
    raise KeyError(f"no certified record with hash prefix {prefix!r}")


def from_lines(coords):
    """LineArrangement from (a, b, c) triples (ints/fractions/strings)."""
    return LineArrangement([ProjectiveLine(*c) for c in coords])


def _clip_line(a, b, c, box):
    """Segment of ax+by+c=0 inside box=(x0,x1,y0,y1), or None."""
    x0, x1, y0, y1 = box
    pts = []
    if abs(b) > 1e-14:
        for X in (x0, x1):
            Y = -(a * X + c) / b
            if y0 - 1e-9 <= Y <= y1 + 1e-9:
                pts.append((X, Y))
    if abs(a) > 1e-14:
        for Y in (y0, y1):
            X = -(b * Y + c) / a
            if x0 - 1e-9 <= X <= x1 + 1e-9:
                pts.append((X, Y))
    if len(pts) < 2:
        return None
    pts = sorted(set(pts))
    return pts[0], pts[-1]


def draw_arrangement(arr, title=None, ax=None, show_doubles=False,
                     pad=0.18, qlo=0.04, qhi=0.96, line_color="#30343a",
                     line_width=1.0, point_scale=30.0, whiten=True):
    """Draw a real line arrangement in the affine chart z = 1.

    Intersection points are computed EXACTLY, then embedded; multiple
    points (m >= 3) are colored by multiplicity.  Lines at infinity
    (a = b = 0) cannot appear in this chart and are counted in the title.
    Complex-field arrangements cannot be drawn in a real chart.
    """
    K = arr.coefficient_field()
    if K is not None and not K.is_real:
        raise ValueError(f"{K.name} arrangement has no real chart z=1")
    pts = arr.intersection_points()
    P, M = [], []
    for p, ls in pts.items():
        w = float(p[2])
        if abs(w) < 1e-12:
            continue                      # point at infinity
        P.append((float(p[0]) / w, float(p[1]) / w))
        M.append(len(ls))
    P = np.array(P); M = np.array(M)

    # optional whitening: an affine change of the z=1 chart spreading the
    # multiple-point cloud isotropically (projectively harmless, much
    # prettier for slope-anisotropic coordinates)
    lines_ab = np.array([[float(v) for v in l.embed()] for l in arr.lines])
    if whiten:
        core = P[M >= 3] if (M >= 3).sum() >= 3 else P
        mu = core.mean(axis=0)
        C = np.cov((core - mu).T) + 1e-9 * np.eye(2)
        evals, evecs = np.linalg.eigh(C)
        W = evecs @ np.diag(evals ** -0.5) @ evecs.T
        P = (P - mu) @ W.T
        newlines = []
        Winv = np.linalg.inv(W)
        for a, b, c in lines_ab:
            ab = np.array([a, b])
            ab_new = ab @ Winv
            c_new = c + ab @ mu
            newlines.append((ab_new[0], ab_new[1], c_new))
        lines_ab = np.array(newlines)
    xs = np.quantile(P[:, 0], [qlo, qhi]); ys = np.quantile(P[:, 1], [qlo, qhi])
    cx, cy = xs.mean(), ys.mean()
    half = max(xs[1] - xs[0], ys[1] - ys[0]) * (0.5 + pad) + 1e-6
    box = (cx - half, cx + half, cy - half, cy + half)

    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(7, 7), dpi=140)
    n_inf = 0
    for (a, b, c) in lines_ab:
        if abs(a) < 1e-12 and abs(b) < 1e-12:
            n_inf += 1
            continue
        seg = _clip_line(a, b, c, box)
        if seg:
            ax.plot([seg[0][0], seg[1][0]], [seg[0][1], seg[1][1]],
                    color=line_color, lw=line_width, alpha=0.85,
                    solid_capstyle="round", zorder=1)
    if show_doubles:
        d = P[M == 2]
        if len(d):
            ax.scatter(d[:, 0], d[:, 1], s=6, color="#b9c2cc", zorder=2)
    for m in sorted(set(M[M >= 3])):
        sel = P[M == m]
        ax.scatter(sel[:, 0], sel[:, 1],
                   s=point_scale * (m - 1.5),
                   color=MULT_COLORS.get(int(m), "#542788"),
                   edgecolors="white", linewidths=0.6, zorder=3,
                   label=f"$m={m}$")
    ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color("#d5dae0")
    if title:
        extra = f"  (+{n_inf} line at infinity)" if n_inf else ""
        ax.set_title(title + extra, fontsize=10)
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    if own:
        return fig
    return None


def draw_gallery(items, ncols=2, figsize=4.6, **kw):
    """items: list of (arr, title).  Grid of drawings."""
    nrows = (len(items) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize * ncols, figsize * nrows),
                             dpi=140)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(items):]:
        ax.axis("off")
    for (arr, title), ax in zip(items, axes):
        draw_arrangement(arr, title=title, ax=ax, **kw)
    fig.tight_layout()
    return fig


## Small classics

In [ ]:
braid = from_lines([(1,0,0), (0,1,0), (0,0,1), (1,-1,0), (1,0,-1), (0,1,-1)])
draw_arrangement(braid, title='braid $A_3$ — free, exponents (1,2,3)', show_doubles=True);


In [ ]:
from known_arrangements import akn13
draw_arrangement(akn13(), title='Abe–Kawanoue–Nozawa $A(13)$ over $\mathbb{Q}(\sqrt3)$ — free (1,6,6), not recursively free');


## Discoveries from the campaigns

Load any certified discovery by its lattice-hash prefix.

In [ ]:
arr, rec = load_by_hash('1a1488047e12')   # eps = 7 record
draw_arrangement(arr, title='$n=27$, exponents $(1,13,13)$, $m=6$, $\epsilon=7$');


In [ ]:
arr20, _ = load_by_hash('b12ce308f784')
draw_arrangement(arr20, title='$n=20$, exponents $(1,9,10)$, $m=5$, $\epsilon=4$');


## Gallery: the four $\epsilon=7$ lattices (pairwise non-isomorphic)

In [ ]:
hashes = ['1a1488047e12', '4d79b15ee813', '65633414ae4b', '19d48696d567']
items = [(load_by_hash(h)[0], h) for h in hashes]
fig = draw_gallery(items, ncols=2)


Figures can be saved with `fig.savefig('name.png', dpi=300, bbox_inches='tight')`.